In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd

kcd_base_path = str(PROCESSED_DIR / 'AEGIS-Dx-KCD/de')

all_kcd_files = glob.glob(os.path.join(kcd_base_path, '**/*.csv'), recursive = True)

excluded_patient_id_set = set()

for file in all_kcd_files:
    try:
        table_df = pd.read_csv(file, dtype = {"pt_id":str}, sep=",", quotechar ='"', lineterminator = '\n')
        if {'depart', 'pt_id'}.issubset(table_df.columns):
            matched_patient_ids = table_df[table_df['depart'].isin(['산부인과', '정신건강의학과'])]['pt_id'].dropna().unique()
            excluded_patient_id_set.update(matched_patient_ids)
    except Exception as e:
        print(f"파일 처리 오류: {file} -> {e}")

excluded_patient_ids = list(excluded_patient_id_set)

print(f"총 {len(excluded_patient_ids)}명")


In [ ]:
import os
import glob
import pandas as pd

table_root = str(PROCESSED_DIR)
table_names = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']


for folder in table_names:
    processed_table_dir = os.path.join(table_root, folder, 'de')

    print(f"\n📂 [{folder}] 처리 시작")

    total_before = 0
    total_after = 0

    csv_files = glob.glob(os.path.join(processed_table_dir, '**/*.csv'), recursive=True)

    for file in csv_files:
        try:
            patient_ids_df = pd.read_csv(file, usecols = ['pt_id'], dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

            if 'pt_id' in patient_ids_df.columns:

                mask = patient_ids_df['pt_id'].isin(excluded_patient_ids)
                num_to_delete = mask.sum()

                if num_to_delete > 0:
                    # 전체 데이터 불러오기
                    complete_table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

                    before_rows = len(complete_table_df)
                    complete_table_df = complete_table_df[~complete_table_df['pt_id'].isin(excluded_patient_ids)]
                    after_rows = len(complete_table_df)

                    total_before += before_rows
                    total_after += after_rows

                    complete_table_df.to_csv(file, index = False, encoding = 'utf-8')

                    print(f"{os.path.relpath(file, processed_table_dir)} - {before_rows - after_rows}행 삭제")
                else:
                    print(f"{os.path.relpath(file, processed_table_dir)} - 변경 없음")
            else:
                print(f"{file} - pt_id 없음 - 건너뜀")
        except Exception as e:
            print(f"오류 발생: {file} -> {e}")
    print(f"{folder} 전체 행 수 변화: {total_before} -> {total_after} ({total_before - total_after}행 삭제)")


In [ ]:
excluded_patients_df = pd.DataFrame(excluded_patient_ids, columns = ['pt_id'])
excluded_patients_df.to_csv(str(OUTPUT_DIR / 'AEGIS_OB_PSY_patients.csv'), index = False, encoding = 'utf-8')


In [ ]:
import os
import glob
import pandas as pd

table_root = str(PROCESSED_DIR / 'AEGIS-Order-Med1/de')
csv_files = glob.glob(os.path.join(table_root, "**/*.csv"), recursive = True)

medication_mapping_parts = []

# med 파일 순회
for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # 필수 컬럼 확인
        required_cols = {"atc_cd", "order_cd", "order_nm"}
        if required_cols.issubset(table_df.columns):
            excluded_orders_df = table_df[table_df['atc_cd'].str.startswith('N05A', na = False)]
            medication_codes_df = excluded_orders_df[['order_cd','order_nm', 'atc_cd']].drop_duplicates()
            if not medication_codes_df.empty:
                medication_mapping_parts.append(medication_codes_df)
                print(f"처리됨: {file} -> {len(medication_codes_df)}건")
            else:
                print(f"'N05A' 코드 없음: {file}")
        else:
            print(f"컬럼 없음: {file}")


    except Exception as e:
        print(f"오류: {file} -> {e}")


# 결과 병합
if medication_mapping_parts:
    excluded_medications_df = pd.concat(medication_mapping_parts, ignore_index = True).drop_duplicates()
    print('\n 최종 매핑 결과')
    print(excluded_medications_df.head())
else:
    print("N05A 관련 매핑 없음")


In [ ]:
#final_mapping_df.to_excel('N05A_mapping.xlsx', index= True)
# 다른 테이블에서 해당 처방코드 제거

excluded_order_codes = excluded_medications_df['order_cd'].dropna().unique().tolist()

table_root = str(PROCESSED_DIR)
table_names = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']

for folder in table_names:
    processed_table_dir = os.path.join(table_root, folder, 'de')
    csv_files = glob.glob(os.path.join(processed_table_dir, '**/*.csv'), recursive = True)

    print(f"\n📂 [{folder}] 처리 중...")

    if not csv_files:
        print(f"파일 없음: {folder}")
        continue

    # 첫 파일 컬럼 확인
    try:
        schema_sample_df = pd.read_csv(csv_files[0], dtype = {"pt_id": str},nrows= 1, sep=",", quotechar ='"', lineterminator = '\n')
        if 'order_cd' not in schema_sample_df.columns:
            print(f"'order_cd' 컬럼 없음 -> {folder} 폴더 전체 건너뜀")
            continue
    except Exception as e:
        print(f"샘플 파일 확인 오류: {csv_files[0]} -> {e}")
        raise

    for file in csv_files:
        try:
            table_df = pd.read_csv(file, dtype = {"pt_id": str}, sep=",", quotechar ='"', lineterminator = '\n')

            before_rows = len(table_df)
            table_df = table_df[~table_df['order_cd'].isin(excluded_order_codes)]
            after_rows = len(table_df)

            if before_rows != after_rows:
                table_df.to_csv(file, index = False, encoding = 'utf-8')
                print(f"삭제 완료: {file} -> {before_rows - after_rows}행 제거")
            else:
                print(f"유지됨: {file}")

        except Exception as e:
            print(f"처리 오류: {file} -> {e}")
            raise


In [ ]:
excluded_medications_df.to_csv(str(OUTPUT_DIR / 'N05A_mapping.csv'),encoding = 'utf-8', index  = False)
#pd.read_csv('N05A_mapping.csv')
